# ML Lab · Module 1, Unit 10: Preparing real data

**Prof. Dr. Mehmet Songur** · ML Lab, Module 1

In this workshop you merge three monthly US series from FRED, handle the missing values correctly and standardise the variables. **Have Gemini write the code**, then **audit it**.

| Table | FRED code | Variable | Starts |
|---|---|---|---|
| `unemployment` | UNRATE | Unemployment rate (%) | 1948 |
| `fed_funds` | FEDFUNDS | Federal funds rate (%) | 1954 |
| `term_spread` | T10Y3MM | 10-year minus 3-month Treasury yield (points) | 1982 |

**Rules**
1. Do not change the **Setup** and **Check** cells; just run them.
2. Generate code with Gemini in the task cells (the Gemini button in a cell or the Gemini panel; the name may vary with Colab's interface).
3. **Read** Gemini's code **before running it**.
4. If a check cell passes, it gives you a **completion code**; paste it into the matching box in ML Lab.

> To keep your own copy: *File → Save a copy in Drive*.

In [ ]:
#@title Setup — run it, do not change it

import io, base64, urllib.request
import numpy as np
import pandas as pd

def fred_download(series_id):
    """Downloads a series from FRED; columns: date, <series_id>. FRED marks missing values with '.';
    they are deliberately turned into NaN and counted."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + series_id
    with urllib.request.urlopen(url) as response:
        df = pd.read_csv(io.BytesIO(response.read()))
    df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    print(f"{series_id}: {len(df)} rows, {df[series_id].isna().sum()} missing, {df['date'].min():%Y-%m} – {df['date'].max():%Y-%m}")
    return df

unemployment = fred_download("UNRATE").rename(columns={"UNRATE": "unemployment"})
fed_funds = fred_download("FEDFUNDS").rename(columns={"FEDFUNDS": "fed_funds"})
term_spread = fred_download("T10Y3MM").rename(columns={"T10Y3MM": "term_spread"})
_copy = {"unemployment": unemployment.copy(), "fed_funds": fed_funds.copy(), "term_spread": term_spread.copy()}
COLS = ["unemployment", "fed_funds", "term_spread"]

def _k(s):
    return base64.b64decode(s).decode()

def _expected():
    d = _copy["unemployment"].merge(_copy["fed_funds"], on="date", how="outer").merge(_copy["term_spread"], on="date", how="outer")
    return d.sort_values("date").reset_index(drop=True)

def check_A(data):
    e = _expected()
    absent = [c for c in ["date"] + COLS if c not in data.columns]
    if absent:
        print("❌ These columns were not found:", absent); return
    if len(data) < len(e):
        print(f"❌ Rows are missing: there should be {len(e)}, there are {len(data)}. An inner join may have been used; did you set how='outer'?"); return
    if len(data) > len(e):
        print(f"❌ Too many rows: there should be {len(e)}, there are {len(data)}. Are there duplicated dates?"); return
    if not pd.Series(data["date"]).is_monotonic_increasing:
        print("❌ The table is not sorted by date."); return
    v = data.reset_index(drop=True)
    for c in COLS:
        if int(v[c].isna().sum()) != int(e[c].isna().sum()):
            print(f"❌ The number of missing values in '{c}' differs from the expected one. Did you fill or drop gaps?"); return
        if not np.allclose(v[c].to_numpy(dtype=float), e[c].to_numpy(dtype=float), equal_nan=True):
            print(f"❌ The values in '{c}' differ from the expected ones."); return
    print("✅ All checks passed. Your completion code: " + _k("TUwxLUJSTC02UFdO"))

def check_B(common, scaled):
    e = _expected().dropna(subset=COLS).reset_index(drop=True)
    absent = [c for c in ["date"] + COLS if c not in common.columns]
    if absent:
        print("❌ 'common' is missing these columns:", absent); return
    if len(common) > len(e):
        print("❌ 'common' is longer than expected: you may have filled gaps (fillna, ffill). The common period is the months in which all three series are observed."); return
    if len(common) < len(e):
        print("❌ 'common' is shorter than expected: some months in which all three series exist were lost."); return
    if common[COLS].isna().any().any():
        print("❌ 'common' still has missing values."); return
    if pd.Timestamp(common["date"].min()) != pd.Timestamp(e["date"].min()):
        print("❌ The start of the common period differs from the expected one."); return
    absent = [c for c in COLS if c not in scaled.columns]
    if absent:
        print("❌ 'scaled' is missing these columns:", absent); return
    if len(scaled) != len(common):
        print("❌ 'scaled' and 'common' should have the same number of rows."); return
    o = scaled[COLS].astype(float).reset_index(drop=True)
    if (o.mean().abs() > 1e-6).any():
        print("❌ The scaled columns do not have mean 0."); return
    if ((o.std() - 1).abs() > 0.01).any():
        print("❌ The scaled columns do not have standard deviation 1. Should std() have been used instead of var()?"); return
    z = (e[COLS] - e[COLS].mean()) / e[COLS].std()
    if not np.allclose(o.to_numpy(), z.to_numpy(), atol=0.01):
        print("❌ The scaled values differ from the expected ones. Is the table sorted by date?"); return
    print("✅ All checks passed. Your completion code: " + _k("TUwxLU9MQy0zS0pW"))

print("Tables ready: unemployment, fed_funds, term_spread")
unemployment.head()

## Task A — Merge

Merge the three tables on the `date` column **so that no month is lost**, sort by date and store the result in a table called `data`. **Do not fill or drop** missing values; print the number missing in each column.

**Sample prompt for Gemini (writing your own is better):**
> There are three pandas tables called unemployment, fed_funds and term_spread; each has a 'date' column (monthly, datetime) and one value column. Merge the three on 'date' with an outer join (how='outer'), sort by date and store the result in 'data'. Do not fill or drop missing values. Print the row count and the number of missing values in each column.

**Audit questions:** What is `how`? Is the row count smaller than the longest series? Are the missing counts consistent with the series' start dates?

In [ ]:
# Task A: generate the code with Gemini here, read it, then run it.

### Check A — run it, do not change it

In [ ]:
check_A(data)

## Task B — Common period and scale

1. From `data`, keep **the months in which all three series exist** and store them in a table called `common`. **Do not fill** gaps: there was no federal funds rate before 1954; filling would mean inventing a past that does not exist.
2. Standardise the three variables in `common` (z = (x − mean) / standard deviation) and store them in a table called `scaled`. The `date` column may stay.

**Audit questions:** Is there a `fillna` or `ffill`? In which month does the common period start? Does the scaling use `std()`? Are the means 0 and the standard deviations 1?

In [ ]:
# Task B: generate the code with Gemini here, read it, then run it.

### Check B — run it, do not change it

In [ ]:
check_B(common, scaled)

## Your audit notes

Before moving to the **audit log** in ML Lab, jot down briefly:
- Which join type did Gemini use? Did you need to change it?
- What did you do with the missing values, and why?
- How did you verify the scaling?